In [1]:
import pandas as pd
import numpy as np
import urllib
from sqlalchemy import create_engine, text

SERVER_NAME = r"BIKKI"  
DATABASE_NAME = "MiniProjectDB"
ODBC_DRIVER = "ODBC Driver 18 for SQL Server"  

params = urllib.parse.quote_plus(
    f"DRIVER={{{ODBC_DRIVER}}};"
    f"SERVER={SERVER_NAME};"
    f"DATABASE={DATABASE_NAME};"
    f"Trusted_Connection=yes;"
    f"TrustServerCertificate=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT @@VERSION AS sql_version, DB_NAME() AS current_db;"))
        row = result.fetchone()
        print(f"--- Connected to: {row.current_db} ---")
except Exception as e:
    print("--- Connection Failed ---")
    print(e)

--- Connected to: MiniProjectDB ---


In [2]:
np.random.seed(42)
n = 1000

data = {
    'ApplicantID': range(1001, 1001 + n),
    'Age': np.random.randint(21, 65, size=n),
    'AnnualIncome': np.random.randint(25000, 150000, size=n),
    'LoanAmount': np.random.randint(5000, 50000, size=n),
    'CreditScore': np.random.randint(480, 850, size=n),
    'EmploymentStatus': np.random.choice(['Employed', 'Self-Employed', 'Unemployed'], size=n, p=[0.75, 0.20, 0.05]),
    'ApplicationDate': pd.date_range(start='2026-01-01', periods=n, freq='h')
}

df = pd.DataFrame(data)
df.head()

,ApplicantID,Age,AnnualIncome,LoanAmount,CreditScore,EmploymentStatus,ApplicationDate
0,1001,59,29018,35811,675,Employed,2026-01-01 00:00:00
1,1002,49,36302,40488,657,Employed,2026-01-01 01:00:00
2,1003,35,92506,13905,486,Self-Employed,2026-01-01 02:00:00
3,1004,63,66157,14127,549,Employed,2026-01-01 03:00:00
4,1005,28,79917,27431,606,Employed,2026-01-01 04:00:00


In [3]:
# The if_exists='replace' ensures you can safely re-run this cell if needed
df.to_sql('raw_loan_applications', con=engine, if_exists='replace', index=False)
print(f"Successfully loaded {len(df)} records into SQL Server.")

Successfully loaded 1000 records into SQL Server.
